In [1]:
!python --version

Python 3.12.10


In [2]:
from pathlib import Path
import numpy as np
import open3d as o3d
import imageio.v3
import mvlm

Jupyter environment detected. Enabling Open3D WebVisualizer.
[Open3D INFO] WebRTC GUI backend enabled.
[Open3D INFO] WebRTCWindowSystem: HTTP handshake server disabled.


In [3]:
# create the model for predicting the landmarks
# The Pipeline can be any of the following:
# * "BU3DFE"
# * "DTU3D"

#file = Path('assets/test2.obj')
file = Path("data/SAPM-MA-03193.obj")
pathToOut = Path("output/")

dm = mvlm.pipeline.create_pipeline("BU3DFE")
landmarks = dm.predict_one_file(file)
np.savetxt((pathToOut / f"{file.stem}.txt").as_posix(), landmarks, delimiter=",")

# if you want to visualize the landmarks on the mesh
#mvlm.utils.VTKViewer(file.as_posix(), landmarks)

Initialising model
Loading checkpoint
Getting device
Render [0] - Prepare 0.000988 s
Render [1] - Setup time:  0.673936 s
Render [2] - Render 0.618986 s
Render [Total]:  1.308135 s
Prediction [Total]:  10.748213 s
Landmarks [0] - From Heatmaps:  0.018258 s
Landmarks [1] - From View Lines:  0.087897 s
Landmarks [2] - Project to Surface:  0.201163 s
Landmarks [Error]:  63095241.857481  mm
Landmarks 3D Total:  12.367668 s


In [4]:
def _copy_mesh(mesh: o3d.geometry.TriangleMesh) -> o3d.geometry.TriangleMesh:
    
    m = o3d.geometry.TriangleMesh(mesh)
    m.vertices = o3d.utility.Vector3dVector(np.asarray(mesh.vertices).copy())
    m.triangles = o3d.utility.Vector3iVector(np.asarray(mesh.triangles).copy())
    if mesh.has_vertex_normals():
        m.vertex_normals = o3d.utility.Vector3dVector(np.asarray(mesh.vertex_normals).copy())
    if mesh.has_triangle_normals():
        m.triangle_normals = o3d.utility.Vector3dVector(np.asarray(mesh.triangle_normals).copy())
    return m

In [5]:
def pointcloud_as_spheres(pcd, radius=0.5, color=[0, 0, 1]):
    spheres = []
    for p in np.asarray(pcd.points):
        sphere = o3d.geometry.TriangleMesh.create_sphere(radius=radius)
        sphere.compute_vertex_normals()
        sphere.translate(p)
        sphere.paint_uniform_color(color)
        spheres.append(sphere)
    return spheres

In [6]:
def generate_gif(meshes, colors, out_path):
    n_frames = 60
    width, height = 800, 600

    #mesh.compute_vertex_normals()
    #mesh.paint_uniform_color([0.949, 0.816, 0.455])
    #spheres = landmarks_as_spheres(landmarks.points, [1,0,0], radius=0.5)
    
    vis = o3d.visualization.Visualizer()
    vis.create_window(visible=True, width=width, height=height)
    #vis.add_geometry(mesh)
    for idx, geom in enumerate(meshes):
        if isinstance(geom, o3d.geometry.TriangleMesh):
            geom = _copy_mesh(geom)
            geom.compute_vertex_normals()

            has_tex = hasattr(geom, "textures") and len(geom.textures) > 0 and geom.triangle_uvs is not None and len(geom.triangle_uvs) > 0
            if not has_tex:
                geom.paint_uniform_color(colors[idx])
            
            vis.add_geometry(geom)

        elif isinstance(geom, o3d.geometry.PointCloud):
            pcd = o3d.geometry.PointCloud(geom)  # copy
            pts = np.asarray(pcd.points)

            sphere_radius = 0.8  # <-- adjust this
            sphere_res = 12      # <-- adjust this (higher = smoother but slower)

            for p in pts:
                sph = o3d.geometry.TriangleMesh.create_sphere(
                    radius=sphere_radius, resolution=sphere_res
                )
                sph.compute_vertex_normals()
                sph.translate(p)
                sph.paint_uniform_color(colors[idx])
                vis.add_geometry(sph)
            
            #geom = o3d.geometry.PointCloud(geom)  # copy
            #geom.paint_uniform_color(colors[idx])
            #vis.add_geometry(geom)
        else:
            raise TypeError(f"Unsupported geometry type: {type(geom)}")

    opt = vis.get_render_option()
    #opt.light_on = True
    opt.background_color = np.array([5.1, 6.7, 9])
    opt.point_size = 10.0
    
    ctr = vis.get_view_control()
    bbox = geom.get_axis_aligned_bounding_box()
    center = bbox.get_center()
    
    radius = np.linalg.norm(bbox.get_extent()) * 1.5
    
    frames = []
    for i in range(n_frames):
        angle = 2 * np.pi * i / n_frames
    
        # --- CAMERA PATH (EDIT HERE) ---
        # orbit around z-axis with slight elevation
        cam_x = center[0] + radius * np.cos(angle)
        cam_y = center[1] + radius * np.sin(angle)
        cam_z = center[2] + radius * 0.3
    
        ctr.set_lookat(center)
        ctr.set_front([center[0] - cam_x,
                       center[1] - cam_y,
                       center[2] - cam_z])
        ctr.set_up([0, 0, 1])      # change if you want different "up"
        ctr.set_zoom(0.8)          # EDIT: closer/further
    
        vis.poll_events()
        vis.update_renderer()
    
        img = vis.capture_screen_float_buffer(False)
        img = (np.asarray(img) * 255).astype(np.uint8)
        frames.append(img)
    
    vis.destroy_window()
    # save GIF
    imageio.mimsave(out_path, frames, fps=10, loop=0)
    
    print(f"Saved rotating GIF as: {out_path}")

In [7]:
mesh = o3d.io.read_triangle_mesh(file, enable_post_processing=True)

L = landmarks.astype(np.float64)
pcd = o3d.geometry.PointCloud()
pcd.points = o3d.utility.Vector3dVector(L)

generate_gif([mesh, pcd], [[1,0,0], [0,0,1]], 'no-PT-BU3DFE.gif')

Saved rotating GIF as: no-PT-BU3DFE.gif


In [8]:
import copy
from pathlib import Path

import numpy as np
import open3d as o3d


def _copy_mesh(mesh):
    return copy.deepcopy(mesh)


def show_interactive_scene(meshes, colors, screenshot_dir="screenshots", window_name="Open3D Viewer"):
    """
    Show the same scene as in generate_gif(), but interactively.

    Controls:
      - Left mouse: rotate
      - Right mouse / wheel: zoom
      - Middle mouse / Shift+left: pan
      - S: save screenshot
      - Q: quit

    Parameters
    ----------
    meshes : list
        List of Open3D geometries (TriangleMesh and/or PointCloud).
    colors : list
        List of RGB colors, one per geometry, e.g. [[1,0,0], [0,1,0], ...]
    screenshot_dir : str or Path
        Directory where screenshots are saved.
    window_name : str
        Window title.
    """
    screenshot_dir = Path(screenshot_dir)
    screenshot_dir.mkdir(parents=True, exist_ok=True)

    vis = o3d.visualization.VisualizerWithKeyCallback()
    vis.create_window(window_name=window_name, width=800, height=600, visible=True)

    added_geometries = []
    world_bbox = o3d.geometry.AxisAlignedBoundingBox()

    for idx, geom in enumerate(meshes):
        if isinstance(geom, o3d.geometry.TriangleMesh):
            g = _copy_mesh(geom)
            g.compute_vertex_normals()

            has_tex = (
                hasattr(g, "textures")
                and len(g.textures) > 0
                and g.triangle_uvs is not None
                and len(g.triangle_uvs) > 0
            )
            if not has_tex:
                g.paint_uniform_color(colors[idx])

            vis.add_geometry(g)
            added_geometries.append(g)
            world_bbox += g.get_axis_aligned_bounding_box()

        elif isinstance(geom, o3d.geometry.PointCloud):
            pcd = o3d.geometry.PointCloud(geom)  # copy
            pts = np.asarray(pcd.points)

            sphere_radius = 0.8
            sphere_res = 12

            for p in pts:
                sph = o3d.geometry.TriangleMesh.create_sphere(
                    radius=sphere_radius, resolution=sphere_res
                )
                sph.compute_vertex_normals()
                sph.translate(p)
                sph.paint_uniform_color(colors[idx])

                vis.add_geometry(sph)
                added_geometries.append(sph)
                world_bbox += sph.get_axis_aligned_bounding_box()

        else:
            raise TypeError(f"Unsupported geometry type: {type(geom)}")

    opt = vis.get_render_option()
    opt.background_color = np.array([5.1, 6.7, 9])
    opt.point_size = 10.0
    opt.light_on = True

    # Fit camera to all displayed geometry
    ctr = vis.get_view_control()
    center = world_bbox.get_center()
    extent = world_bbox.get_extent()
    radius = np.linalg.norm(extent)

    ctr.set_lookat(center)
    ctr.set_front([0.3, -0.8, -0.4])
    ctr.set_up([0, 0, 1])
    ctr.set_zoom(0.8)

    # Screenshot hotkey
    counter = {"n": 0}

    def save_screenshot(v):
        out_path = screenshot_dir / f"screenshot_{counter['n']:03d}.png"
        v.capture_screen_image(str(out_path), do_render=True)
        print(f"Saved screenshot: {out_path}")
        counter["n"] += 1
        return False

    # Optional: reset camera hotkey
    def reset_camera(v):
        ctr = v.get_view_control()
        ctr.set_lookat(center)
        ctr.set_front([0.3, -0.8, -0.4])
        ctr.set_up([0, 0, 1])
        ctr.set_zoom(0.8)
        print("Camera reset.")
        return False

    vis.register_key_callback(ord("S"), save_screenshot)
    vis.register_key_callback(ord("R"), reset_camera)

    print("Interactive viewer opened.")
    print("Controls:")
    print("  Mouse = rotate / zoom / pan")
    print("  S     = save screenshot")
    print("  R     = reset camera")
    print("  Q     = quit")

    vis.run()
    vis.destroy_window()

In [9]:
show_interactive_scene(
    [mesh, pcd],
    [[1, 0, 0], [0, 0, 1], [0, 1, 0]],
    screenshot_dir="screenshots"
)

Interactive viewer opened.
Controls:
  Mouse = rotate / zoom / pan
  S     = save screenshot
  R     = reset camera
  Q     = quit
